In [1]:
import os
from pathlib import Path

os.environ["HF_HOME"] = "/kaggle/temp/hf"
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("fitcheck")
    print("HF token loaded")
except Exception as error:
    print("No HF token:", error)
    print("Every model in the grid below is open, so this is not fatal.")

HF token loaded


In [3]:
import os

REPO = "/kaggle/working/fitcheck"

if not os.path.isdir(f"{REPO}/.git"):
    !git clone -q https://github.com/Anassbzdd/fitcheck.git /kaggle/working/fitcheck

%cd /kaggle/working/fitcheck
!git pull --ff-only || echo "git pull failed (dirty tree or diverged) -- using the code already on disk"
!git log --oneline -1

/kaggle/working/fitcheck
Already up to date.
1e72b19 (HEAD -> main, origin/main, origin/HEAD) fix:overhead key, c_overhead eq


In [4]:
!pip install -q -e .
!pip uninstall -q -y torchao


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fitcheck-llm (pyproject.toml) ... done


In [5]:
import argparse
import contextlib
import importlib
import io
import os
import sys

REPO = "/kaggle/working/fitcheck"
os.environ["PYTHONPATH"] = REPO
if f"{REPO}/scripts" not in sys.path:
    sys.path.insert(0, f"{REPO}/scripts")


def preflight() -> tuple[bool, str]:
    sweep = importlib.reload(importlib.import_module("calibration_sweep"))
    captured = io.StringIO()
    with contextlib.redirect_stdout(captured):
        ok = sweep.preflight(argparse.Namespace(quant="nf4")) is not None
    print(captured.getvalue().rstrip())
    return ok, captured.getvalue()


ok, report = preflight()

if not ok and "torchvision does not match torch" in report:
    print("\n--- dropping the mismatched torchvision/torchaudio, then retrying ---\n")
    !pip uninstall -q -y torchvision torchaudio
    ok, report = preflight()

if not ok and "bitsandbytes is NOT installed" in report:
    print("\n--- installing bitsandbytes (no -U, so torch is left alone) ---\n")
    !pip install -q bitsandbytes
    ok, report = preflight()

print()
print("STACK OK -- run the sweep." if ok else "STACK BROKEN -- fix the above first.")


PREFLIGHT FAILED -- `--quant nf4` needs bitsandbytes and it did not import.

  ModuleNotFoundError: No module named 'bitsandbytes'

bitsandbytes is NOT installed, and `--quant nf4` / `--quant int8` load the base
model through it. Kaggle and Colab ship torch, transformers and peft but not this
one, so an unquantized grid runs on a bare image and a quantized grid cannot.

    pip install bitsandbytes

Without `-U`. A plain install leaves the image's torch alone -- bitsandbytes
only requires a torch, and the image already has one that satisfies it -- while
`-U` upgrades torch itself and takes torchvision down with it (see the torchvision
note above). No kernel restart is needed: every row runs in a fresh subprocess.

Or take the unquantized grid instead, which needs nothing installed:

    --quant none   (and drop --tag nf4)

--- installing bitsandbytes (no -U, so torch is left alone) ---

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.0 MB/s eta 0:00:00:00:0100:01
Tesla T4 (

In [6]:
# VALIDATION 1/4 -- predictions, registered BEFORE any measurement.
import json, subprocess, sys, time
from pathlib import Path

REPO = "/kaggle/working/fitcheck"
OUT = Path("/kaggle/working/validation"); OUT.mkdir(parents=True, exist_ok=True)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from fitcheck.config_parser import fetch_model_config
from fitcheck.estimator import TrainingConfig, estimate
from fitcheck.gpu_db import get_gpu
from fitcheck.overhead_db import get_overhead_profile

MODELS = [
    "meta-llama/Llama-3.2-1B-Instruct",          # Llama-3, tied embeddings, 128k vocab (GATED)
    "deepseek-ai/deepseek-coder-1.3b-instruct",  # full MHA, small vocab
    "stabilityai/stablelm-2-1_6b",               # full MHA (32 kv heads), 100k vocab
    "meta-llama/Llama-3.2-3B-Instruct",          # GQA 24/8 (GATED)
    "mistralai/Mistral-7B-Instruct-v0.3",        # GQA 32/8, 7B
    "Qwen/Qwen2.5-7B-Instruct",                  # GQA 28/4, 152k vocab, tightest fit
]
GPU, SEQ, RANK, QUANT = "t4", 1024, 16, "nf4"
t4 = get_gpu(GPU)

def predict(model_id, bs, kernel):
    cfg = fetch_model_config(model_id)
    training = TrainingConfig(
        precision="fp16", quantization=QUANT, double_quant=False, optimizer="adamw",
        lora_rank=RANK, lora_targets=["q_proj", "k_proj", "v_proj", "o_proj"],
        batch_size=bs, seq_len=SEQ, grad_checkpoint=True, flash_attn=(kernel == "flash"))
    return cfg, estimate(cfg, training, t4)

plan, skipped = [], []
print(f"{'model':<34}{'kern':>6}{'bs':>4}{'W_base':>8}{'A_act':>8}{'C_ovh':>7}"
      f"{'TOTAL':>8}{'maxbs':>6}  role")
for model_id in MODELS:
    try:
        cfg, ref = predict(model_id, 1, "flash")
    except Exception as error:
        skipped.append((model_id, f"{type(error).__name__}: {error}"))
        print(f"{model_id.split('/')[-1][:33]:<34}  SKIPPED -- {str(error)[:60]}")
        continue
    rows = [(1, "flash", "accuracy"), (1, "eager", "accuracy")]
    max_bs = ref.max_batch_size
    if max_bs >= 1:
        rows.append((max_bs, "flash", "max-must-fit"))
        rows.append((max_bs + 1, "flash", "max+1-must-OOM"))
    for bs, kernel, role in rows:
        _, rep = predict(model_id, bs, kernel)
        profile = get_overhead_profile(GPU, kernel == "flash", QUANT)
        plan.append(dict(
            model_id=model_id, batch_size=bs, seq_len=SEQ, kernel=kernel, role=role,
            lora_rank=RANK, quantization=QUANT, predicted_total_mib=rep.total_mib,
            predicted_tensors_mib=rep.total_mib - rep.overhead_mib,
            predicted_overhead_mib=rep.overhead_mib,
            predicted_weight_mib=rep.weight_mib, predicted_activation_mib=rep.activation_mib,
            predicted_max_batch_size=max_bs, fits=rep.total_mib <= t4.usable_mib,
            overhead_profile=profile.source, model_config=cfg.__dict__.copy()))
        print(f"{model_id.split('/')[-1][:33]:<34}{kernel:>6}{bs:>4}{rep.weight_mib:>8,.0f}"
              f"{rep.activation_mib:>8,.0f}{rep.overhead_mib:>7,.0f}{rep.total_mib:>8,.0f}"
              f"{max_bs:>6}  {role}")

(OUT / "predictions.json").write_text(json.dumps(plan, indent=1), encoding="utf-8")
print(f"\n{len(plan)} rows registered, {len(skipped)} models skipped -> {OUT}/predictions.json")
for model_id, why in skipped:
    print(f"  SKIPPED {model_id}: {why}")
print("\nNOTE: the two meta-llama models are GATED. If they skipped, accept the licence on")
print("their model pages with the same account as your HF token, then re-run this cell.")


model                               kern  bs  W_base   A_act  C_ovh   TOTAL maxbs  role


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

Llama-3.2-1B-Instruct              flash   1   1,524   2,132    350   4,059     5  accuracy
Llama-3.2-1B-Instruct              eager   1   1,524   2,132    558   4,267     5  accuracy
Llama-3.2-1B-Instruct              flash   5   1,524  10,660  1,189  13,425     5  max-must-fit
Llama-3.2-1B-Instruct              flash   6   1,524  12,792  1,398  15,766     5  max+1-must-OOM


config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

deepseek-coder-1.3b-instruct       flash   1   1,156     696    326   2,274    14  accuracy
deepseek-coder-1.3b-instruct       eager   1   1,156     696    375   2,323    14  accuracy
deepseek-coder-1.3b-instruct       flash  14   1,156   9,744  2,731  13,727    14  max-must-fit
deepseek-coder-1.3b-instruct       flash  15   1,156  10,440  2,916  14,608    14  max+1-must-OOM


...

Llama-3.2-3B-Instruct              flash   1   3,016   2,340    410   5,906     4  accuracy
Llama-3.2-3B-Instruct              eager   1   3,016   2,340    490   5,985     4  accuracy
Llama-3.2-3B-Instruct              flash   4   3,016   9,360  1,217  13,733     4  max-must-fit
Llama-3.2-3B-Instruct              flash   5   3,016  11,700  1,487  16,342     4  max+1-must-OOM


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

Mistral-7B-Instruct-v0.3           flash   1   4,769   1,024    537   6,538     6  accuracy
Mistral-7B-Instruct-v0.3           eager   1   4,769   1,294    495   6,766     6  accuracy
Mistral-7B-Instruct-v0.3           flash   6   4,769   6,144  2,517  13,638     6  max-must-fit
Mistral-7B-Instruct-v0.3           flash   7   4,769   7,168  2,913  15,058     6  max+1-must-OOM


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

Qwen2.5-7B-Instruct                flash   1   7,659   2,768    558  11,139     1  accuracy
Qwen2.5-7B-Instruct                eager   1   7,659   2,768    580  11,162     1  accuracy
Qwen2.5-7B-Instruct                flash   1   7,659   2,768    558  11,139     1  max-must-fit
Qwen2.5-7B-Instruct                flash   2   7,659   5,536    975  14,324     1  max+1-must-OOM

24 rows registered, 0 models skipped -> /kaggle/working/validation/predictions.json

NOTE: the two meta-llama models are GATED. If they skipped, accept the licence on
their model pages with the same account as your HF token, then re-run this cell.


In [7]:
# VALIDATION 2/4 -- the measurements. Model-major, so each model downloads once.
import argparse, importlib, json, os, subprocess, sys, time
from pathlib import Path

REPO = "/kaggle/working/fitcheck"
OUT = Path("/kaggle/working/validation")
if f"{REPO}/scripts" not in sys.path:
    sys.path.insert(0, f"{REPO}/scripts")
sweep = importlib.reload(importlib.import_module("calibration_sweep"))

plan = json.loads((OUT / "predictions.json").read_text(encoding="utf-8"))
if sweep.preflight(argparse.Namespace(quant="nf4")) is None:
    raise SystemExit("preflight failed -- fix the stack first")

# group by model so a 7B is fetched once, not four times
by_model: dict[str, list[dict]] = {}
for row in plan:
    by_model.setdefault(row["model_id"], []).append(row)

clock = time.time()
for model_id, rows in by_model.items():
    print(f"\n=== {model_id} ===", flush=True)
    for row in rows:
        bs, kernel, role = row["batch_size"], row["kernel"], row["role"]
        slug = sweep._slug(model_id, bs, row["seq_len"], kernel, f"-{row['quantization']}")
        target = OUT / f"{slug}.json"
        if target.exists():
            print(f"  [{role}] bs={bs} {kernel}  (already done)"); continue

        command = sweep._row_args(
            model_id, bs, row["seq_len"], kernel,
            argparse.Namespace(gpu="t4", quant=row["quantization"],
                               precision="fp16", lora_r=row["lora_rank"]))
        started = time.time()
        print(f"  [{role}] bs={bs} {kernel} ...", flush=True)
        result = subprocess.run(command, capture_output=True, text=True)
        took = time.time() - started

        if result.returncode != 0:
            stderr = result.stderr or ""
            oom = "out of memory" in stderr.lower()
            target.write_text(json.dumps({
                "model_id": model_id, "run": {"batch_size": bs, "kernel": kernel,
                "seq_len": row["seq_len"], "quantization": row["quantization"]},
                "failed": True, "oom": oom,
                "stderr_tail": stderr.strip().splitlines()[-5:]}, indent=1), encoding="utf-8")
            verdict = "OOM" if oom else "FAILED (not memory)"
            print(f"      {verdict} in {took:.0f}s"
                  f"{'  <-- EXPECTED' if (oom and role == 'max+1-must-OOM') else ''}")
            if not oom:
                for line in stderr.strip().splitlines()[-4:]:
                    print(f"        {line[:150]}")
            continue

        payload = json.loads(result.stdout)
        payload["role"] = role
        target.write_text(json.dumps(payload, indent=1), encoding="utf-8")
        m, e = payload["measured"], payload["error_pct"]
        print(f"      ok {took:.0f}s  process {m['process_mib']:,.0f} MiB"
              f"  tensors {e['tensors']:+.1f}%  process {e['process']:+.1f}%")

print(f"\ntotal {time.time() - clock:.0f}s")


Tesla T4 (sm_75, 14,912 MiB) | torch 2.10.0+cu128 | transformers 5.0.0 | peft 0.19.1 | bitsandbytes 0.50.2

=== meta-llama/Llama-3.2-1B-Instruct ===
  [accuracy] bs=1 flash ...
      ok 25s  process 4,283 MiB  tensors -0.4%  process -5.2%
  [accuracy] bs=1 eager ...
      ok 18s  process 4,521 MiB  tensors -0.5%  process -5.6%
  [max-must-fit] bs=5 flash ...
      ok 26s  process 13,421 MiB  tensors -0.4%  process +0.0%
  [max+1-must-OOM] bs=6 flash ...
      OOM in 15s  <-- EXPECTED

=== deepseek-ai/deepseek-coder-1.3b-instruct ===
  [accuracy] bs=1 flash ...
      ok 34s  process 2,345 MiB  tensors -1.1%  process -3.0%
  [accuracy] bs=1 eager ...
      ok 17s  process 2,391 MiB  tensors -1.4%  process -2.8%
  [max-must-fit] bs=14 flash ...
      ok 53s  process 13,487 MiB  tensors -1.1%  process +1.8%
  [max+1-must-OOM] bs=15 flash ...
      ok 58s  process 14,293 MiB  tensors -1.1%  process +2.2%

=== stabilityai/stablelm-2-1_6b ===
  [accuracy] bs=1 flash ...
      FAILED (not memo

In [8]:
# VALIDATION 3/4 -- predicted vs measured, with the <5% / 5-10% / >10% verdict.
import json
from pathlib import Path

OUT = Path("/kaggle/working/validation")
plan = {(r["model_id"], r["batch_size"], r["kernel"]): r
        for r in json.loads((OUT / "predictions.json").read_text(encoding="utf-8"))}

def band(pct):
    a = abs(pct)
    return "<5%" if a < 5 else ("5-10%" if a < 10 else ">10%")

acc, feas = [], []
for path in sorted(OUT.glob("*.json")):
    if path.name == "predictions.json":
        continue
    row = json.loads(path.read_text(encoding="utf-8"))
    run = row["run"]
    key = (row["model_id"], run["batch_size"], run["kernel"])
    reg = plan.get(key)
    if reg is None:
        continue
    if row.get("failed"):
        feas.append((reg, None, row.get("oom", False)))
        continue
    measured = row["measured"]["process_mib"]
    err = 100.0 * (reg["predicted_total_mib"] - measured) / measured
    if reg["role"] == "accuracy":
        acc.append((reg, measured, err, row["error_pct"]["tensors"]))
    else:
        feas.append((reg, measured, False))

RULE = "=" * 78
print(RULE); print("A. VRAM ACCURACY -- bs=1, seq=1024, nf4, LoRA r=16"); print(RULE)
print(f"{'model':<30}{'kern':>6}{'predicted':>11}{'measured':>10}{'process':>9}{'tensors':>9}{'band':>8}")
for reg, measured, err, terr in sorted(acc, key=lambda t: -abs(t[2])):
    print(f"{reg['model_id'].split('/')[-1][:29]:<30}{reg['kernel']:>6}"
          f"{reg['predicted_total_mib']:>11,.0f}{measured:>10,.0f}{err:>+8.1f}%"
          f"{terr:>+8.1f}%{band(err):>8}")

if acc:
    errs = [e for _, _, e, _ in acc]
    worst_under = min(errs)
    counts = {b: sum(1 for e in errs if band(e) == b) for b in ("<5%", "5-10%", ">10%")}
    print(f"\n  n={len(errs)}   mean |err| {sum(abs(e) for e in errs)/len(errs):.1f}%"
          f"   worst over {max(errs):+.1f}%   worst under {worst_under:+.1f}%")
    print(f"  bands: <5% = {counts['<5%']}   5-10% = {counts['5-10%']}   >10% = {counts['>10%']}")
    overall = ("<5%" if max(abs(e) for e in errs) < 5
               else "5-10%" if max(abs(e) for e in errs) < 10 else ">10%")
    print(f"  ==> WORST-CASE BAND: {overall}")
    print(f"  ==> OOM direction (under-prediction) worst: {worst_under:+.1f}%")

print()
print(RULE); print("B. MAX FEASIBLE BATCH SIZE -- does the predicted ceiling hold?"); print(RULE)
print(f"{'model':<34}{'bs':>4}{'role':>16}{'outcome':>12}{'verdict':>10}")
right = wrong = 0
for reg, measured, oom in sorted(feas, key=lambda t: t[0]["model_id"]):
    want_fit = reg["role"] == "max-must-fit"
    got_fit = measured is not None
    ok = want_fit == got_fit
    right, wrong = right + ok, wrong + (not ok)
    outcome = f"{measured:,.0f} MiB" if got_fit else ("OOM" if oom else "failed")
    print(f"{reg['model_id'].split('/')[-1][:33]:<34}{reg['batch_size']:>4}"
          f"{reg['role']:>16}{outcome:>12}{'PASS' if ok else 'FAIL':>10}")
print(f"\n  max_batch_size correct on {right}/{right + wrong} checks")


A. VRAM ACCURACY -- bs=1, seq=1024, nf4, LoRA r=16
model                           kern  predicted  measured  process  tensors    band
Qwen2.5-7B-Instruct            eager     11,162    12,481   -10.6%    -0.1%    >10%
Llama-3.2-3B-Instruct          flash      5,906     6,391    -7.6%    +0.0%   5-10%
Llama-3.2-3B-Instruct          eager      5,985     6,371    -6.1%    -0.1%   5-10%
Llama-3.2-1B-Instruct          eager      4,267     4,521    -5.6%    -0.5%   5-10%
Llama-3.2-1B-Instruct          flash      4,059     4,283    -5.2%    -0.4%   5-10%
deepseek-coder-1.3b-instruct   flash      2,274     2,345    -3.0%    -1.1%     <5%
deepseek-coder-1.3b-instruct   eager      2,323     2,391    -2.8%    -1.4%     <5%
Mistral-7B-Instruct-v0.3       eager      6,766     6,897    -1.9%    +1.1%     <5%
Mistral-7B-Instruct-v0.3       flash      6,538     6,513    +0.4%    -0.9%     <5%

  n=9   mean |err| 4.8%   worst over +0.4%   worst under -10.6%
  bands: <5% = 4   5-10% = 4   >10% = 1
  ==

In [9]:
# VALIDATION 4/4 -- one archive file, with enough provenance to keep it.
import json, subprocess, sys, time
from pathlib import Path

REPO = "/kaggle/working/fitcheck"
OUT = Path("/kaggle/working/validation")
if REPO not in sys.path:
    sys.path.insert(0, REPO)

probe = subprocess.run([sys.executable, "-c",
    "import json,torch,transformers,peft;"
    "import bitsandbytes as bnb;"
    "p=torch.cuda.get_device_properties(0);"
    "print(json.dumps({'torch':torch.__version__,'transformers':transformers.__version__,"
    "'peft':peft.__version__,'bitsandbytes':bnb.__version__,'gpu':p.name,"
    "'capability':'sm_%d%d'%(p.major,p.minor),'total_mib':round(p.total_memory/1024**2)}))"],
    capture_output=True, text=True)
stack = json.loads(probe.stdout) if probe.returncode == 0 else {"error": probe.stderr[-400:]}

import fitcheck
commit = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"],
                        capture_output=True, text=True).stdout.strip()

rows = []
for path in sorted(OUT.glob("*.json")):
    if path.name in ("predictions.json", "validation-archive.json"):
        continue
    rows.append(json.loads(path.read_text(encoding="utf-8")))

archive = {
    "_provenance": {
        "what": "Out-of-sample validation of the task 9.3 C_overhead calibration. Six models, "
                "none used to fit it, LoRA r=16 (the calibration was fitted at r=32). Two tests "
                "per model: VRAM error at bs=1 seq=1024 nf4 both kernels, and whether the "
                "predicted max_batch_size holds (max fits, max+1 OOMs). Predictions were "
                "registered in predictions.json BEFORE any measurement.",
        "measured_on": time.strftime("%Y-%m-%d %H:%M:%S"),
        "stack": stack,
        "fitcheck_version": fitcheck.__version__,
        "fitcheck_commit": commit,
        "gpu_key": "t4",
        "grid": "bs=1 seq=1024 nf4 fp16 AdamW r=16 [q,k,v,o] grad-checkpoint, eager + flash",
    },
    "predictions": json.loads((OUT / "predictions.json").read_text(encoding="utf-8")),
    "runs": rows,
}
(OUT / "validation-archive.json").write_text(json.dumps(archive, indent=1), encoding="utf-8")
print(f"archived {len(rows)} rows, commit {commit[:8]}, {stack.get('gpu','?')}")

!cd /kaggle/working && rm -f validation.zip && zip -qr validation.zip validation
!ls -l --time-style=+%H:%M:%S /kaggle/working/validation.zip
print("built at", time.strftime("%H:%M:%S"), "-- refresh the Output panel if it shows older")


archived 23 rows, commit 1e72b19a, Tesla T4
-rw-r--r-- 1 root root 34420 23:34:01 /kaggle/working/validation.zip
built at 23:34:01 -- refresh the Output panel if it shows older


In [10]:
# README 1/2 -- recompute the affected Process err column from archived measurements.
import json, sys
from pathlib import Path

REPO = "/kaggle/working/fitcheck"
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from fitcheck.config_parser import ModelConfig, fetch_model_config
from fitcheck.estimator import TrainingConfig, estimate
from fitcheck.gpu_db import get_gpu

ARCHIVE = Path(REPO) / "data" / "measurements"

def model_config(row):
    # 2026-09-15 and later embed it; the 2026-09-01 archive does not, so fetch config.json
    if "model_config" in row:
        return ModelConfig(**row["model_config"])
    return fetch_model_config(row["model_id"])

def rescore(row):
    run = row["run"]
    training = TrainingConfig(
        precision=run["precision"], quantization=run["quantization"],
        double_quant=run.get("double_quant", False), optimizer=run["optimizer"],
        lora_rank=run["lora_rank"], lora_targets=list(run["lora_targets"]),
        batch_size=run["batch_size"], seq_len=run["seq_len"],
        grad_checkpoint=run["grad_checkpoint"], flash_attn=run["kernel"] == "flash")
    report = estimate(model_config(row), training, get_gpu(run["gpu_key"]))
    measured_process = row["measured"]["peak_reserved_mib"] + row["measured"]["cuda_context_mib"]
    tensors = report.total_mib - report.overhead_mib
    return dict(
        model=row["model_id"].split("/")[-1], bs=run["batch_size"], seq=run["seq_len"],
        kernel=run["kernel"], quant=run["quantization"], ckpt=run["grad_checkpoint"],
        predicted_tensors=tensors, measured_tensors=row["measured"]["peak_allocated_mib"],
        tensors_err=100.0 * (tensors - row["measured"]["peak_allocated_mib"])
                    / row["measured"]["peak_allocated_mib"],
        predicted_total=report.total_mib, measured_process=measured_process,
        process_err=100.0 * (report.total_mib - measured_process) / measured_process,
        old_process_err=row.get("error_pct", {}).get("process"))

scored = []
for name in ("t4-qlora-2026-09-01.json", "t4-sweep-2026-09-15.json",
             "t4-phase3-2026-09-16.json"):
    path = ARCHIVE / name
    if not path.exists():
        print(f"  missing {name} -- git pull?"); continue
    for row in json.loads(path.read_text(encoding="utf-8"))["runs"]:
        src = row["run"].get("source_file", "")
        if "-r2" in src or "-r3" in src:
            continue                       # bit-identical repeats
        scored.append((name, rescore(row)))

print(f"{len(scored)} archived rows re-scored with the CURRENT code\n")
print(f"{'model':<24}{'bs':>3}{'seq':>6}{'kern':>6}{'quant':>6}"
      f"{'tensors':>9}{'process NOW':>13}{'process WAS':>13}")
for _, s in sorted(scored, key=lambda t: (t[1]["quant"], t[1]["kernel"], t[1]["model"])):
    was = f"{s['old_process_err']:+.1f}%" if s["old_process_err"] is not None else "--"
    print(f"{s['model'][:23]:<24}{s['bs']:>3}{s['seq']:>6}{s['kernel']:>6}{s['quant']:>6}"
          f"{s['tensors_err']:>+8.1f}%{s['process_err']:>+12.1f}%{was:>13}")

t = [s["tensors_err"] for _, s in scored]
p = [s["process_err"] for _, s in scored]
print(f"\nSUMMARY (for the README table)")
print(f"  tensors  max abs {max(abs(x) for x in t):.1f}%   mean abs {sum(abs(x) for x in t)/len(t):.1f}%")
print(f"  process  max abs {max(abs(x) for x in p):.1f}%   mean abs {sum(abs(x) for x in p)/len(p):.1f}%")
print(f"  process  worst UNDER-prediction (the OOM direction): {min(p):+.1f}%")


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

59 archived rows re-scored with the CURRENT code

model                    bs   seq  kern quant  tensors  process NOW  process WAS
Qwen2.5-1.5B-Instruct     2  1024 eager   nf4    -0.3%        -2.4%           --
Qwen2.5-1.5B-Instruct     2  1024 eager   nf4    -0.4%        -4.6%        -0.6%
Qwen2.5-1.5B-Instruct     1  2048 eager   nf4    -0.6%        -1.7%        -1.0%
SmolLM2-1.7B              4  1024 eager   nf4    -0.3%        +0.4%           --
SmolLM2-1.7B              4  1024 eager   nf4    -0.3%        -0.5%       -15.4%
SmolLM2-1.7B              1  2048 eager   nf4    +0.7%       +12.0%        +2.0%
SmolLM2-135M              1  4096 eager   nf4    -3.2%        -0.3%       -21.0%
SmolLM2-135M              2   512 eager   nf4    -1.0%        -5.2%       +20.6%
SmolLM2-360M              1  4096 eager   nf4    -4.6%        -7.4%       -25.5%
TinyLlama-1.1B-Chat-v1.   2   512 eager   nf4    -0.8%        -0.2%           --
TinyLlama-1.1B-Chat-v1.   2  1024 eager   nf4    -1.0%     

In [11]:
# README 2/2 -- the single unarchived row from the checkpointing-ON table.
!cd /kaggle/working/fitcheck && python scripts/measure.py JackFram/llama-160m \
    --gpu t4 --quant none --precision fp16 --no-lora \
    --batch-size 2 --seq-len 1024 --grad-checkpoint --json \
    > /kaggle/working/validation/llama-160m-fullft-bs2-seq1024-eager-none.json

!cd /kaggle/working && python -c "import json;d=json.load(open('validation/llama-160m-fullft-bs2-seq1024-eager-none.json'));e=d['error_pct'];print('tensors %+.1f%%  process %+.1f%%' % (e['tensors'], e['process']))"


config.json: 100%|█████████████████████████████| 555/555 [00:00<00:00, 2.34MB/s]
measure.py: 2 GPUs visible; measuring device 0 (Tesla T4) only. This is by design -- fitcheck predicts single-device memory, so a sharded or DDP run would not be comparable to the prediction.
model.safetensors: 100%|██████████████████████| 650M/650M [00:03<00:00, 170MB/s]
Loading weights: 100%|█| 111/111 [00:00<00:00, 703.57it/s, Materializing param=m
generation_config.json: 100%|███████████████████| 107/107 [00:00<00:00, 477kB/s]
tensors +0.4%  process -15.2%
